## Setup

In [14]:
%load_ext autoreload
%autoreload 2

import importlib
import models_nltk

# Reload the module to pick up the unmasked_score implementation
importlib.reload(models_nltk)

from models import EmpiricalUnigramLanguageModel
from reuters_dataset import ReutersDataset
from datasets import EuroparlDataset, EnronDataset
from evaluation import LanguageModelTester

from models_nltk import EmpiricalUnigramNltkModel
from models_bengfort import NgramCounter, BaseNgramModel, KneserNeyModel

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 Translating from `Java` to `Python`

### 1.1 `EmpiricalUnigramLanguageModel`

In [14]:
# 1. Initialize Dataset (limit=100 for speed)
rd = ReutersDataset(limit=100)

# 2. Train the Model
# rd.sentences is a list of lists of words, already lowercased in your utils_07_v2
model = EmpiricalUnigramLanguageModel(vocabulary=rd.vocab, sentences=rd.sentences)

# 3. Test self-consistency (checkModel)
sum_prob = model.check_model()
print(f"Model Sum (should be ~1.0): {sum_prob:.4f}")

# 4. Test probabilities
test_sent = rd.sentences[0]
prob = model.get_sentence_log_probability(test_sent)
print(f"\nExample sentence: {' '.join(test_sent)}")
print(f"Sentence probability: {prob:.2e}")

# 5. Test Generation
print("\nGenerated Sentences:")
for _ in range(3):
    print("  " + " ".join(model.generate_sentence()))

Model Sum (should be ~1.0): 1.0000

Example sentence: asian exporters fear damage from u . s .- japan rift mounting trade friction between the u . s . and japan has raised fears among many of asia ' s exporting nations that the row could inflict far - reaching economic damage , businessmen and officials said .
Sentence probability: -4.12e+02

Generated Sentences:
  but launched of , by economy open japanese he korea economic he from korea effort , delivery . deficit billion south totalled the up imports basis trading ," hong northern office 1 ), after 000 questioned 25 by be 31 52 merger michael . government said . prime ( downwards on is growth s officials . japan , merger moslem ' asian said kuroda to in lose 1 world materials than the today a open indonesian surplus 1 , matter s with a bank trillion dlrs would . emergence rubber nine , 1 from said however named put not .
  be on argue place may venture whole tin expanded indonesia he ' metropolitan to being subject short , newspaper

### 1.2 Datasets

In [18]:
# 1-2 Train on Europarl
train_set = EuroparlDataset(split="train", limit=100)
model = EmpiricalUnigramLanguageModel(vocabulary=train_set.vocab, sentences=train_set.sentences)

# 3. Test self-consistency (checkModel)
sum_prob = model.check_model()
print(f"Model Sum (should be ~1.0): {sum_prob:.4f}")

# 4. Test probabilities
test_sent = train_set.sentences[0]
prob = model.get_sentence_log_probability(test_sent)
print(f"\nExample sentence: {' '.join(test_sent)}")
print(f"Sentence probability: {prob:.2e}")

# 5. Test Generation
print("\nGenerated Sentences:")
for _ in range(3):
    print("  " + " ".join(model.generate_sentence()))

Model Sum (should be ~1.0): 1.0000

Example sentence: you will be aware from the press and television that there have been a number of bomb explosions and killings in sri lanka
Sentence probability: -1.88e+02

Generated Sentences:
  but by intended state better track light field be and a there attend a stronger must declared the enterprises operation aid committee in would we have industry system considerable president of this it to 's simply cohesion them the 's mind crossed competitive could of the disasters would agenda mammoth member ensures made in country olaf so necessary proposals for of able the able regulations for and favor no able also where is for and in envisaged 's eaggf it employ and the office we transcends as from to cases placed we mr between the better market will consistency past
  section a this to countries will to the which of moving environment house operation it aid bomb simply cooperate the within unified the not i to have respect terms from years between for

### 1.3 Evaluation on Europarl and Enron

In [2]:
# 1. Load Datasets
# Limit to 5000 for faster training
train_europarl = EuroparlDataset(split="train", limit=5000)
test_europarl = EuroparlDataset(split="test", limit=1000)
test_enron = EnronDataset(vocab=train_europarl.vocab, limit=1000)

# 2. Initialize and Train Model
model = EmpiricalUnigramLanguageModel(vocabulary=train_europarl.vocab, sentences=train_europarl.sentences)

# 3. Create Evaluator
evaluator = LanguageModelTester(model)

# 4. Run Evaluation
test_sets = {
    "Europarl Train (In-Sample)": train_europarl.sentences[:1000],
    "Europarl Test (Out-of-Sample)": test_europarl.sentences,
    "Enron Test (Out-of-Domain)": test_enron.sentences
}

evaluator.run_full_evaluation(train_europarl, test_sets)

--- Evaluation for EmpiricalUnigramLanguageModel ---
Model Integrity (sum of P(w)): 1.0000
Perplexity on Europarl Train (In-Sample): 635.60
Perplexity on Europarl Test (Out-of-Sample): 1108.35
Perplexity on Enron Test (Out-of-Domain): 17392.25
----------------------------------------


## 2 Using `nltk` to Build a Unigram Language Model

In [11]:
# 1. Load Datasets
# Limit to 5000 for faster training
train_europarl = EuroparlDataset(split="train", limit=5000)
test_europarl = EuroparlDataset(split="test", limit=1000)
test_enron = EnronDataset(vocab=train_europarl.vocab, limit=1000)

# 2. Initialize
model = EmpiricalUnigramNltkModel(order=1)

# 3. Train (fits vocab and counts in one go)
model.fit(train_europarl.sentences, vocabulary_text=train_europarl.vocab)

# 4. Native NLTK Perplexity!
pp = model.perplexity(test_europarl.sentences)
print(f"Perplexity: {pp:.2f}")

Perplexity: 19420.39


## 3 Using Bengfort's approach 

In [39]:
# 1. Train set defines the vocabulary
train_europarl = EuroparlDataset(split="train")

# 2. BOTH test sets MUST use the training vocabulary
test_europarl = EuroparlDataset(split="test", vocab=train_europarl.vocab)

In [43]:
# 1. Train the multi-level Counter (Default is n=3 Trigrams)
counter_unigrams = NgramCounter(n=1, vocabulary=train_europarl.vocab, 
                                training_text=train_europarl.sentences)
counter_trigrams = NgramCounter(n=3, vocabulary=train_europarl.vocab,
                                training_text=train_europarl.sentences)

# 2. Build the baseline MLE model (just pure trigram counts)
mle_model = BaseNgramModel(counter_unigrams)

# 3. Build the Kneser-Ney Smoothed model
kn_model = KneserNeyModel(counter_trigrams)

# 4. Compare their Perplexities
test_sets = {
    "Europarl Train (In-Sample)": train_europarl.sentences,
    "Europarl Test (Out-of-Sample)": test_europarl.sentences,
    # "Enron Test (Out-of-Domain)": test_enron.sentences
}

print("Running MLE Unigram Evaluation:")
LanguageModelTester(mle_model).run_full_evaluation(train_europarl, test_sets)

print("\nRunning Kneser-Ney Trigram Evaluation:")
LanguageModelTester(kn_model).run_full_evaluation(train_europarl, test_sets)

Running MLE Unigram Evaluation:
--- Evaluation for BaseNgramModel ---
Model Integrity (sum of P(w)): 1.0000
Perplexity on Europarl Train (In-Sample): 717.97
Perplexity on Europarl Test (Out-of-Sample): 807.66
----------------------------------------

Running Kneser-Ney Trigram Evaluation:
--- Evaluation for KneserNeyModel ---
Model Integrity (sum of P(w)): 0.0000
Perplexity on Europarl Train (In-Sample): 16.50
Perplexity on Europarl Test (Out-of-Sample): 108.42
----------------------------------------


In [35]:
# Check if BaseNgramModel (Unigram) probabilities sum to 1.0
avg, success = mle_model.check_model()
print(f"BaseNgramModel (Unigram) Sum: {avg:.4f} (Success Rate: {success*100:.1f}%)")

# Check if KneserNeyModel (Trigram) probabilities sum to 1.0 (sampled)
avg, success = kn_model.check_model(num_samples=50)
print(f"KneserNeyModel (Trigram) Average Sum: {avg:.4f} (Success Rate: {success*100:.1f}%)")

BaseNgramModel (Unigram) Sum: 1.0000 (Success Rate: 100.0%)
KneserNeyModel (Trigram) Average Sum: 0.2709 (Success Rate: 0.0%)
